# API Domonstration


### Import libs


In [ ]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np

from uavcalibration.datasets import VisLocDataset
from uavcalibration.calibration import Calibration
from uavcalibration.transform import *

### Read uav data


In [ ]:
dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
uav_data = dataset[0]

### Coarse calibration (perspective transform)


In [ ]:
calibration = Calibration(uav_data.uav_image)
calibration.coarse_calibrate(**asdict(uav_data))
src_shape = uav_data.uav_image.shape
calibration.transform.adjust_shape(src_shape=(src_shape[1], src_shape[0]))

### Calculate image area


In [ ]:
h, w = calibration.uav_image.shape[:2]
corner = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float64)
corner_utm = cv2.perspectiveTransform(
    corner.reshape(1, -1, 2), calibration.transform.combined.mat
).reshape(-1, 2)
transformer = Transformer.from_crs(
    calibration.transform.crs.crs, "epsg:4326", always_xy=True
)
corner_lonlat = np.stack(transformer.transform(corner_utm[:, 0], corner_utm[:, 1]), 1)
corner_lonlat = np.stack([corner_lonlat.min(0), corner_lonlat.max(0)])
print(corner_lonlat.ravel())

### Fetch satellite image and transform


In [ ]:
# TODO: query satellite image
satellite_image = uav_data.satellite_image
satellite_crs = CRSTransform(uav_data.satellite_transform, uav_data.satellite_crs)

### Fine calibration process (image matching)


In [ ]:
calibration.fine_calibrate(satellite_image=satellite_image, satellite_crs=satellite_crs)

### Get target transform


In [ ]:
print(calibration.transform.combined)